<a href="https://colab.research.google.com/github/karenl7/engr550-internal/blob/main/Module_3_HW_robot_charging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Robot charging problem with gradient descent and log-barrier

In [ ]:
import jax
import jax.numpy as jnp
from typing import Callable
from matplotlib import pyplot as plt

First, implement a function that performs vanilla gradient descent.
A simple test function is provided to help you verify your implementation.

In [ ]:
def gradient_descent(
    func: Callable[[jnp.ndarray], jnp.ndarray],
    guess: jnp.ndarray,
    learning_rate: float = 0.01,
    num_steps: int = 100,
):
    """
    Perform gradient descent to find the minimum of a provided function
    Args:
        func: The function to minimize.
        guess: The initial guess.
        learning_rate: The learning rate (step size).
        num_steps: The number of iterations.
    Returns:
        guesses: The list of guesses.
        values: The list of function values.
        sol: The final guess.
    """

    grad_func = jax.jit(jax.grad(func)) # JIT compile the gradient function for efficiency
    guesses = [guess] # Initialize the list of guesses with the initial guess
    values = [func(guess)] # Initialize the list of function values with the value at the initial guess

    # TODO: implement gradient descent using the loop below.
    # Keep the initial guess and value already stored above. Each iteration
    # should add exactly one new iterate and one new objective value.

    for _ in range(num_steps):
        # TODO: complete one gradient descent step.
        # Evaluate grad_func at the most recent entry in guesses.
        # Move that iterate in the direction of decreasing function value,
        # scaling the gradient by learning_rate.
        # Append the new iterate to guesses and evaluate func at that same
        # iterate before appending its value to values.
        # Use JAX array operations so the routine works for vectors or matrices.
        # Replace the placeholder below with your implementation.
        raise NotImplementedError("Implement one gradient descent step.")
        # end TODO

    # return the list of guesses, the list of function values, and the final guess
    return jnp.array(guesses), jnp.array(values), guesses[-1]



In [ ]:
# try it out on a simple function
def test_function(x):
    return jnp.sum(x**2)

guess = jnp.array([1.0, 1.0]) * 3

xs, cs, x_star = gradient_descent(test_function, guess, learning_rate=0.1, num_steps=100)
plt.plot(cs)
plt.title(f"cost vs. iteration, final cost = {cs[-1]:.4f}")
plt.xlabel("iteration")
plt.ylabel("cost")
plt.grid()

Now, we define the problem parameters of the robot charging problem

In [ ]:
N, T = 4, 8
c = jnp.array([0.18, 0.16, 0.14, 0.13, 0.15, 0.20, 0.24, 0.22], dtype=jnp.float32)
E = jnp.array([12.0, 10.0, 9.0, 11.0], dtype=jnp.float32)
r_max = jnp.array([3.0, 2.5, 2.0, 3.5], dtype=jnp.float32)
P_max = jnp.array([8.0, 8.0, 7.0, 7.0, 7.0, 6.0, 6.0, 6.0], dtype=jnp.float32)



Here, implement the following functions needed to construct the objective function with log-barrier constraints.
Read the docstring to understand the expected inputs and outputs of each function.

In [ ]:
def original_objective(x, gamma):
    """
    The original objective function without the log barrier.
    Args:
        x: The decision variable (charging schedule). Size: (N, T)
        gamma: The weight for the smoothness term.
    Returns:
        The value of the original objective function.
    """
    # TODO: implement the original objective function.
    # Define energy_cost by totaling charging energy across robots in each
    # time slot and weighting those totals by the corresponding prices in c.
    # Define smoothness as the sum of squared changes between consecutive
    # time slots for each robot. Do not take differences between robots.
    # Both quantities should be scalars; the return statement combines them.
    # Replace the placeholder below with your implementation.
    raise NotImplementedError("Define energy_cost and smoothness.")
    # end TODO
    return energy_cost + gamma * smoothness

def log_barrier_function(y, beta):
    """
    The log barrier function for the constraints.
    phi(y) = -1/beta * sum(log(-y_i + 1e-9)) for all i where y_i < 0
    NOTE: add a small constant to avoid log(0) which is undefined.

    Args:
        y: The constraint values (should be negative).
            It can be a vector, in which case, the log barrier function is applied element-wise and summed.
        beta: The barrier parameter.
    Returns:
        The value of the log barrier function.
    """
    # TODO: implement the log barrier function.
    # Translate the formula in the docstring into JAX operations.
    # For strictly feasible inputs, negate y to obtain positive log arguments
    # and include the small constant specified above for numerical stability.
    # Sum over all entries and scale by the barrier parameter with the
    # appropriate sign. Return a scalar that grows near a constraint boundary.
    # Replace the placeholder below with your implementation and return value.
    raise NotImplementedError("Implement the scalar log barrier penalty.")
    # end TODO

def constraints(x):
    """
    Compute the constraint functions in the form g(x) <= 0.
    Args:
        x: The decision variable (charging schedule). Size: (N, T)
    Returns:
        A list of constraint functions.
    """
    # TODO: implement the constraints.
    # Define g1, g2, g3, and g4 so each entry is <= 0 when its constraint
    # is satisfied (and strictly < 0 at an interior point for the log barrier).
    # g1: enforce nonnegative charging energy for every entry of x.
    # g2: ensure each robot receives at least its required overnight energy E.
    # g3: enforce each robot's hourly charging-rate limit in r_max.
    # g4: enforce the station's total capacity P_max in each time slot.
    # Choose the summation axis carefully for robot totals versus slot totals.
    # Reshape per-robot limits as a column when broadcasting across time slots.
    # The code below flattens and concatenates the four constraint arrays.
    # Replace the placeholder below with your implementation.
    raise NotImplementedError("Define the four constraint arrays g1 through g4.")
    # end TODO


    gs = [g1, g2, g3, g4]
    return jnp.concatenate([g.flatten() for g in gs])



def unconstrained_objective(x, lam, beta):
    """
    Unconstrained objective function for the barrier method.
    Args:
        x: The decision variable (charging schedule).
        lam: The weight for the smoothness term.
        beta: The barrier parameter.
    Returns:
        The value of the unconstrained objective function.
    """
    gs = constraints(x)
    return original_objective(x, lam) + log_barrier_function(gs, beta)



Now, we need to come up with an initial guess that is feasible.

In [ ]:
# a simple heuristic for an initial guess that satisfies the constraints
# this is a relatively simple problem where a heuristic like this works.
# but in general, you may need to use a more sophisticated method to find a feasible starting point.

# Just run this cell
scale_factor = 0.95
x0 = ((r_max / jnp.sum(r_max, axis=0))[:, None] * P_max[None, :]) * scale_factor
gs = constraints(x0)  # compute the constraint values for the initial guess
print(f"Constraint values: {gs}")  # print the constraint values
print(f"All constraints satisfied: {(gs < 0).all()}")  # check if the initial guess satisfies all constraints

Now, let's perform gradient descent on the objective function with log-barrier constraints incoroporated it in.
We also want to slowly increase $\beta$ as the gradient descent iteration progresses.

In [ ]:
lam = 0.1 # weight for the smoothness term
beta_schedule = jnp.linspace(1.0, 1000, 50) # schedule for the barrier parameter. Feel free to change this if you want to experiment with different schedules.
learning_rate = 0.01 # learning rate for gradient descent
num_steps = 2000 # number of steps for gradient descent
x_star = x0     # initial guess for the optimization
xs_list = []    # list to store the optimization path for each beta
cs_list = []    # list to store the cost for each beta


for beta in beta_schedule:
    objective = lambda x: unconstrained_objective(x, lam, beta)
    xs, cs, x_star = gradient_descent(objective, x_star, learning_rate=learning_rate, num_steps=num_steps)
    xs_list.append(xs) # collect the optimization path for this beta
    cs_list.append(cs) # collect the cost for this beta

    final_cost = original_objective(x_star, lam)
    energy_cost = original_objective(x_star, 0.0)
    print(f"beta: {beta:.3f}, \t final cost: {final_cost:.3f}, \t energy cost: {energy_cost:.3f}")

# compute the final cost and energy cost for the last x_star
final_cost = original_objective(x_star, lam)
energy_cost = original_objective(x_star, 0.0)


# combine the optimization paths and costs for all beta values into single arrays for plotting
xs = jnp.concatenate(xs_list)
cs = jnp.concatenate(cs_list)


# Check if the optimized solution satisfies all constraints
gs = constraints(x_star)
print(f"Constraint values for optimized solution: {gs}")  # print the constraint values
print(f"All constraints satisfied: {(gs < 0).all()}")  # check if the optimized solution satisfies all constraints


In [ ]:
# helper function to plot the charging schedule
def plot_charging_schedule(x_sol, title):
    """
    Plot the charging schedule for each robot over time.
    """

    for i in range(N):
        plt.plot(range(T), x_sol[i, :], marker='o', label=f'Robot {i+1}')
        plt.hlines(r_max[i], 0, T-1, color=f"C{i}", linestyles='dashed')
    plt.plot(jnp.sum(x_sol, axis=0), marker='o', color='black', label='Total')
    plt.plot(range(T), P_max, color='black', linestyle='dotted', label='Station Cap')

    plt.title(title)
    plt.xlabel('Time Slot')
    plt.ylabel('Charging Energy (kWh)')
    plt.xticks(range(T))
    plt.ylim(0, max(P_max) + 1)
    plt.grid()
    plt.legend(ncol=2)


In [ ]:
plt.figure(figsize=(15, 6))
plt.subplot(1,2,1)
for x in xs[::num_steps]:
    [plt.plot(range(T), x[i, :], color=f"C{i}", alpha=0.1) for i in range(N)]
    plt.plot(jnp.sum(x, axis=0), color='black', alpha=0.1)
plot_charging_schedule(x_star, "Optimized Charging Schedule. Optimal Cost = {:.4f}, Energy Cost = {:.4f}".format(final_cost, energy_cost))


plt.subplot(1,2,2)
plt.plot(cs[::num_steps])
plt.title("Cost vs. Iteration")
plt.grid()

plt.tight_layout()